### **Adaptação código Aula 03:**
- SnowballC (wordsStem(), stopwords)
- Índice Inverso
- Tabela de frequência de termo por documento

### Testando Corpus Original:

In [34]:
if (!require(SnowballC)) install.packages("SnowballC")
library(SnowballC)

In [35]:
docs <- c(
  d1 = "Recuperacao de Informacao: ORDENA documentos, por relevancia!",
  d2 = "O modelo de espaco-vetorial representa documentos (como vetores).",
  d3 = "BM25 e um modelo probabilistico de ranqueamento de texto.",
  d4 = "Aprendizado estatistico fundamenta a recuperacao moderna.",
  d5 = "O indice invertido acelera a busca em muitos documentos.",
  d6 = "Embeddings capturam a semantica de palavras e documentos.",
  d7 = "A avaliacao mede a relevancia dos resultados da busca.",
  d8 = "Ciencia de dados combina estatistica e programacao."
)

In [36]:
processar_texto <- function(txt) {
    txt <- tolower(txt)
    txt <- gsub("[[:punct:]]", "", txt)
    tokens <- unlist(strsplit(txt, "\\s+"))
    tokens <- tokens[tokens != ""]
  
    stopwords_multilingual <- unique(c(
        stopwords::stopwords("pt", "snowball")))

    tokens <- tokens[
    !tokens %in% stopwords_multilingual]

    tokens <- wordStem(tokens, language = "portuguese")     # Separando RADICAL da palavra.
    return(tokens)
}

In [37]:
postings <- list()

for (doc_id in names(docs)) {
  tokens <- processar_texto(docs[doc_id])
  tokens_unicos <- unique(tokens)
  for (tok in tokens_unicos) {
    postings[[tok]] <- c(postings[[tok]], doc_id) }
}

# Contagem de aparições por doc. de um >> PSEUDO-RADICAL <<:

print(postings[["recuperaca"]])    # Retorna em quantos "docs"/unidades de texto 
print(postings[["acel"]])          # o PSEUDO-RADICAL "document" aparece.
print(postings[["captur"]])

[1] "d1" "d4"
[1] "d5"
[1] "d6"


In [38]:
preparar_consulta <- function(texto) {
  processar_texto(texto)
}

buscar_AND <- function(consulta, indice = postings) {
  termos <- preparar_consulta(consulta)
  if (length(termos) == 0) return(character(0))
  
  listas <- indice[termos]
  listas <- listas[!sapply(listas, is.null)]
  if (length(listas) == 0) return(character(0))
  Reduce(intersect, listas)
}

buscar_OR <- function(consulta, indice = postings) {
  termos <- preparar_consulta(consulta)
  if (length(termos) == 0) return(character(0))
  
  listas <- indice[termos]
  listas <- listas[!sapply(listas, is.null)]
  if (length(listas) == 0) return(character(0))
  Reduce(union, listas)
}

In [39]:
cat("Resultado_1:", buscar_AND(consulta = "modelo probabilistico"),"\n") # 1 dos docs. possuem AMBOS os pseudo-radicais.
cat("Resultado_2:", buscar_OR(consulta = "modelo probabilistico"))       # 2 documentos possue ALGUM dos 2 pseudo-radicais.

Resultado_1: d3 
Resultado_2: d2 d3

In [40]:
tabela_indice <- data.frame(
  Radical = names(postings),                                     # 1ª coluna (pseudo-radical listado)
  Frequencia_Docs = lengths(postings),                           # 2ª coluna (quantos docs)
  Documentos = I(sapply(postings, paste, collapse = ", "))       # 3ª coluna (d1, d2...)
)

tabela_indice <- tabela_indice[order(tabela_indice$Radical), ]
print(View(head(tabela_indice, 10)))

,Radical,Frequencia_Docs,Documentos
,<chr>,<int>,<I<chr>>
acel,acel,1,d5
aprendiz,aprendiz,1,d4
avaliaca,avaliaca,1,d7
bm25,bm25,1,d3
busc,busc,2,"d5, d7"
captur,captur,1,d6
cienc,cienc,1,d8
combin,combin,1,d8
dad,dad,1,d8


          Radical Frequencia_Docs     Documentos
acel         acel               1             d5
aprendiz aprendiz               1             d4
avaliaca avaliaca               1             d7
bm25         bm25               1             d3
busc         busc               2         d5, d7
captur     captur               1             d6
cienc       cienc               1             d8
combin     combin               1             d8
dad           dad               1             d8
document document               4 d1, d2, d5, d6


### Testando Dataframe de Bertioga oriundo do Web-Scraping:

In [53]:
# Caminho do arquivo
caminho <- "C:/Users/Ivan/Documents/Pasta-Documentos-PC-antigo/GITHUB-Meus-Repositorios/PesquisaPI3_2026/data/raw/noticias_santos.csv"

# Ler o CSV
noticias <- read.csv(caminho, stringsAsFactors = FALSE, fileEncoding = "UTF-8")

# Pegar APENAS a coluna titulo
docs_noticias <- noticias$titulo
names(docs_noticias) <- paste0("n", 1:length(docs_noticias))

# Ver os 3 primeiros
head(docs_noticias, 3)

n1 
        "CVV busca voluntários em Santos para ampliar atendimento no Setembro Amarelo" 
                                                                                    n2 
"‘Castelo’, funcionário conhecido do Restaurante Almeida, morre aos 82 anos em Santos" 
                                                                                    n3 
     "Santos abre pré-inscrição para escolas municipais em 2027; veja quem pode fazer"

In [ ]:
postings_noticias <- list()
for (doc_id in names(docs_noticias)) {
  tokens <- processar_texto(docs_noticias[doc_id])
  tokens_unicos <- unique(tokens)
  for (tok in tokens_unicos) {
    postings_noticias[[tok]] <- c(postings_noticias[[tok]], doc_id) }
}

print(postings_noticias[["sant"]])                # Retorna em quantos "docs"/unidades de texto 
print(postings_noticias[["funcionari"]])          # o PSEUDO-RADICAL "document" aparece.

 [1] "n1"  "n2"  "n3"  "n4"  "n5"  "n6"  "n7"  "n8"  "n9"  "n10" "n11" "n12"
[13] "n13" "n15" "n16" "n17" "n18" "n19" "n20"
NULL


In [74]:
cat("Resultado_AND:", buscar_AND(consulta = "Santos", indice = postings_noticias), "\n")
cat("Resultado_OR:", buscar_OR(consulta = "morre ampliar", indice = postings_noticias))

Resultado_AND: n1 n2 n3 n4 n5 n6 n7 n8 n9 n10 n11 n12 n13 n15 n16 n17 n18 n19 n20 
Resultado_OR: n2 n1

In [73]:
tabela_noticias <- data.frame(
  Radical = names(postings_noticias),
  Frequencia_Docs = lengths(postings_noticias),
  Documentos = I(sapply(postings_noticias, paste, collapse = ", "))
)
tabela_noticias <- tabela_noticias[order(tabela_noticias$Radical), ]
View(tabela_noticias)

,Radical,Frequencia_Docs,Documentos
,<chr>,<int>,<I<chr>>
10,10,1,n17
100,100,2,"n4, n8"
2026,2026,1,n9
2027,2027,2,"n3, n12"
3,3,1,n14
7,7,1,n15
80,80,1,n6
82,82,1,n2
abre,abre,2,"n3, n20"
